# Sukhothai drought prototype: Earth Engine data pull (tambon × month)

Exports monthly tambon-level drought variables for all 86 tambons of Sukhothai to Google Drive (folder `sukhothai_drought`).

| Export | Dataset | Period | Variables |
|---|---|---|---|
| `chirps_monthly` | CHIRPS daily | 1981 → now | monthly rainfall (mm) → SPI |
| `era5land_monthly` | ERA5-Land monthly | 1981 → ~2–3 months ago | soil moisture L1/L2, PET, ET, rain, temp → SPEI, SM anomaly |
| `modis_ndvi_monthly` | MOD13Q1 | 2000 → now | NDVI (all land + cropland only) → VCI |
| `modis_lst_monthly` | MOD11A2 | 2000 → now | day LST (°C) → TCI, VHI |
| `s2_water_monthly` | Sentinel-2 SR | 2017 → now | open-water area, clear-sky area → small reservoir/pond storage proxy |
| `static_layers` | WorldPop, ESA WorldCover, JRC GSW | static | population, cropland, permanent water |

**How to run:** Runtime → Run all. You'll be asked to (1) sign in to Google for Earth Engine, (2) type your Cloud project ID, (3) upload `sukhothai_tambon_gee.geojson`.

**Security:** sign-in uses your own Google account via the standard Earth Engine prompt. Do not paste keys, tokens or service-account JSON into this notebook, and do not save the notebook with outputs that show tokens.

Requirement: a Google Cloud project registered for Earth Engine (noncommercial use is free): https://code.earthengine.google.com/register

In [ ]:
import ee, json
ee.Authenticate()   # opens Google sign-in; no keys stored in the notebook
PROJECT_ID = input('Earth Engine Cloud project ID: ').strip()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine ready')

In [ ]:
from google.colab import files
up = files.upload()                       # choose sukhothai_tambon_gee.geojson
gj = json.loads(next(iter(up.values())).decode('utf-8'))
feats = [ee.Feature(ee.Geometry(f['geometry']), {k: f['properties'][k] for k in ('pcode', 'tambon', 'amphoe')})
         for f in gj['features']]
tambons = ee.FeatureCollection(feats)
print('tambons:', tambons.size().getInfo())   # expect 86
REGION = tambons.geometry().bounds().buffer(5000)

In [ ]:
import datetime
FOLDER = 'sukhothai_drought'
TODAY = datetime.date.today()
END = f'{TODAY.year}-{TODAY.month:02d}-01'   # months before the current one

def month_starts(start, end=END):
    s, e = ee.Date(start), ee.Date(end)
    n = e.difference(s, 'month').round()
    return ee.List.sequence(0, n.subtract(1)).map(lambda i: s.advance(i, 'month'))

def zonal(img, scale, reducer=ee.Reducer.mean(), tile_scale=2):
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM')
    fc = img.reduceRegions(collection=tambons, reducer=reducer, scale=scale, tileScale=tile_scale)
    return fc.map(lambda f: f.set('date', date).setGeometry(None))

def export(fc, name, cols):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name, folder=FOLDER,
                                      fileNamePrefix=name, fileFormat='CSV',
                                      selectors=['pcode', 'tambon', 'amphoe', 'date'] + cols)
    t.start(); print('started', name)
    return t

tasks = []

## 1. CHIRPS monthly rainfall (for SPI-1/3/6)

In [ ]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
def chirps_month(d):
    d = ee.Date(d)
    img = chirps.filterDate(d, d.advance(1, 'month')).sum().rename('rain_mm')
    return img.set('system:time_start', d.millis()).set('n_days', chirps.filterDate(d, d.advance(1, 'month')).size())
chirps_m = ee.ImageCollection(month_starts('1981-01-01').map(chirps_month)).filter(ee.Filter.gt('n_days', 0))
tasks.append(export(chirps_m.map(lambda i: zonal(i, 5566)).flatten(), 'chirps_monthly', ['rain_mm']))

## 2. ERA5-Land monthly (soil moisture, PET, ET → SPEI, soil-moisture anomaly)

In [ ]:
era_bands = ['volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'total_precipitation_sum',
             'potential_evaporation_sum', 'total_evaporation_sum', 'temperature_2m']
era_new = ['sm_l1', 'sm_l2', 'era_rain_m', 'pet_m', 'et_m', 't2m_k']
era = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate('1981-01-01', END)
       .select(era_bands, era_new))
tasks.append(export(era.map(lambda i: zonal(i, 11132)).flatten(), 'era5land_monthly', era_new))

## 3. MODIS NDVI monthly (all land + cropland only, for VCI)

In [ ]:
crop = ee.ImageCollection('ESA/WorldCover/v200').first().eq(40)
ndvi16 = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
def ndvi_month(d):
    d = ee.Date(d)
    m = ndvi16.filterDate(d, d.advance(1, 'month')).mean().multiply(0.0001)
    img = m.rename('ndvi').addBands(m.updateMask(crop).rename('ndvi_crop'))
    return img.set('system:time_start', d.millis()).set('n', ndvi16.filterDate(d, d.advance(1, 'month')).size())
ndvi_m = ee.ImageCollection(month_starts('2000-03-01').map(ndvi_month)).filter(ee.Filter.gt('n', 0))
tasks.append(export(ndvi_m.map(lambda i: zonal(i, 250)).flatten(), 'modis_ndvi_monthly', ['ndvi', 'ndvi_crop']))

## 4. MODIS land surface temperature monthly (for TCI → VHI)

In [ ]:
lst8 = ee.ImageCollection('MODIS/061/MOD11A2').select('LST_Day_1km')
def lst_month(d):
    d = ee.Date(d)
    img = lst8.filterDate(d, d.advance(1, 'month')).mean().multiply(0.02).subtract(273.15).rename('lst_c')
    return img.set('system:time_start', d.millis()).set('n', lst8.filterDate(d, d.advance(1, 'month')).size())
lst_m = ee.ImageCollection(month_starts('2000-03-01').map(lst_month)).filter(ee.Filter.gt('n', 0))
tasks.append(export(lst_m.map(lambda i: zonal(i, 1000)).flatten(), 'modis_lst_monthly', ['lst_c']))

## 5. Sentinel-2 open-water area monthly (ponds / small reservoirs proxy)

Water = MNDWI > 0 on cloud-free pixels (SCL classes 4–7, 11 kept as clear land/water). Reports water area and clear-sky area in km²; use `water_km2 / clear_km2` when clear coverage is incomplete. Expect some noise from shadows and flooded paddy in the wet season; the anomaly is most meaningful for Oct–May.

In [ ]:
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(REGION)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)))
def s2_prep(i):
    scl = i.select('SCL')
    clear = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7)).Or(scl.eq(11))
    mndwi = i.normalizedDifference(['B3', 'B11'])
    return mndwi.gt(0).rename('water').updateMask(clear).addBands(clear.selfMask().rename('clear'))
def s2_month(d):
    d = ee.Date(d)
    col = s2.filterDate(d, d.advance(1, 'month')).map(s2_prep)
    water = col.select('water').max()        # water seen in any clear scene that month
    clear = col.select('clear').max().unmask(0)
    pa = ee.Image.pixelArea().divide(1e6)
    img = water.unmask(0).multiply(pa).rename('water_km2').addBands(clear.multiply(pa).rename('clear_km2'))
    return img.set('system:time_start', d.millis()).set('n', col.size())
s2_m = ee.ImageCollection(month_starts('2017-04-01').map(s2_month)).filter(ee.Filter.gt('n', 0))
tasks.append(export(s2_m.map(lambda i: zonal(i, 20, ee.Reducer.sum(), tile_scale=8)).flatten(),
                    's2_water_monthly', ['water_km2', 'clear_km2']))

## 6. Static layers (exposure): population, cropland, permanent water

In [ ]:
pa = ee.Image.pixelArea().divide(1e6)
pop = ee.ImageCollection('WorldPop/GP/100m/pop').filter(ee.Filter.eq('country', 'THA')).filter(ee.Filter.eq('year', 2020)).first().rename('pop_2020')
wc = ee.ImageCollection('ESA/WorldCover/v200').first()
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
static = (pop.addBands(wc.eq(40).multiply(pa).rename('crop_km2'))
             .addBands(wc.eq(50).multiply(pa).rename('builtup_km2'))
             .addBands(gsw.select('occurrence').gte(75).unmask(0).multiply(pa).rename('perm_water_km2'))
             .addBands(gsw.select('occurrence').gt(0).unmask(0).multiply(pa).rename('any_water_km2'))
             .set('system:time_start', ee.Date('2020-01-01').millis()))
tasks.append(export(zonal(static, 30, ee.Reducer.sum(), tile_scale=4), 'static_layers',
                    ['pop_2020', 'crop_km2', 'builtup_km2', 'perm_water_km2', 'any_water_km2']))

## Monitor tasks (re-run this cell until all are COMPLETED; usually 10–60 min)

In [ ]:
for t in tasks:
    s = t.status(); print(f"{s['description']:<22} {s['state']:<10} {s.get('error_message', '')}")

When all tasks are **COMPLETED**, the six CSV files are in Google Drive → `sukhothai_drought/`. Tell Claude they're ready; it can read them through the Google Drive connector (or attach the CSVs in the chat).